### Semantic Chunking
- SemanticChunker is a document splitter that uses embedding similarity between sentences to decide chunk boundaries.

- It ensures that each chunk is semantically coherent and not cut off mid-thought like traditional character/token splitters.

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [4]:
## Initialize the model
model = SentenceTransformer('all-MiniLM-L6-v2')

## Sample text
text="""
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

## Step 1 : Split into sentences
sentences = [s.strip() for s in text.split("\n") if s.strip()]

### sstep 2: Embed each setence
embeddings = model.encode(sentences)

# Step 3: Initialize parameters
threshold = 0.7  # control chunk tightness
chunks = []
current_chunk = [sentences[0]]

## Step 4: Semantic grouping based on threshold

for i in range(1, len(sentences)):
    sim = cosine_similarity(
        [embeddings[i - 1]],
        [embeddings[i]]
    )[0][0]

    if sim >= threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]

# Append the last chunk
chunks.append(" ".join(current_chunk))

# Output the chunks
print("\n📌 Semantic Chunks:")
for idx, chunk in enumerate(chunks):
    print(f"\nChunk {idx+1}:\n{chunk}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7204.78it/s]



📌 Semantic Chunks:

Chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

Chunk 2:
You can create chains, agents, memory, and retrievers.

Chunk 3:
The Eiffel Tower is located in Paris.

Chunk 4:
France is a popular tourist destination.


### RAG Pipeline Modular Coding

In [20]:
from langchain_ollama import OllamaEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_core.runnables  import RunnableLambda, RunnableMap
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

### Custom Semantic Chunker

In [26]:
class ThresholdSematicChunker:
    def __init__(self, model_name = "qwen3-embedding:8b", threshold = 0.5):
        self.model = OllamaEmbeddings(model=model_name)
        self.threshold = threshold

    def split(self, text: str):
        sentences = [s.strip() for s in text.split(".") if s.strip()]
        
        # Handle edge case where text is empty or doesn't contain valid sentences
        if not sentences:
            return []
            
        print(sentences)
        embeddings = self.model.embed_documents(sentences)
        chunks = []
        
        # 1. Initialize the chunk with the FIRST sentence
        current_chunk = [sentences[0]]
        
        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i-1]], [embeddings[i]])[0][0]
            if sim >= self.threshold:
                # 2. Append the CURRENT sentence (not the previous one)
                current_chunk.append(sentences[i])
            else:
                # 3. Save the chunk and start a new one with the CURRENT sentence
                chunks.append(". ".join(current_chunk) + ".")
                current_chunk = [sentences[i]]
        
        # Append whatever is left in the final chunk
        if current_chunk:
            chunks.append(". ".join(current_chunk) + ".")
            
        return chunks

    def split_documents(self, docs):
        result=[]
        for doc in docs:
            for chunk in self.split(doc.page_content):
                result.append(Document(page_content=chunk, metadata=doc.metadata))

        return result
        

In [15]:
# Sample text
sample_text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

doc = Document(page_content = sample_text)
print(doc.page_content)


LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.



In [27]:
### Chunking
chunker = ThresholdSematicChunker(threshold=0.7)
chunks = chunker.split_documents([doc])
chunks

['LangChain is a framework for building applications with LLMs', 'Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone', 'You can create chains, agents, memory, and retrievers', 'The Eiffel Tower is located in Paris', 'France is a popular tourist destination']


[Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers.'),
 Document(metadata={}, page_content='The Eiffel Tower is located in Paris.'),
 Document(metadata={}, page_content='France is a popular tourist destination.')]

In [29]:
embedding = OllamaEmbeddings(model="qwen3-embedding:8b")
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever()

#  Prompt Template

In [30]:
template = """Answer the question based on the following context:

{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n\n{context}\n\nQuestion: {question}\n')

### LCEL Chain With retrieval

In [31]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model = "deepseek-r1:14b",
    temperature = 0.4
)

rag_chain=(
    RunnableMap(
        {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"],  
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

# --- 8. Run Query ---
query = {"question": "What is LangChain used for?"}
result = rag_chain.invoke(query)

print(result)



LangChain is a framework designed for developing applications that utilize Large Language Models (LLMs) and other AI tools. It enables the integration of these models with services like OpenAI and Pinecone, allowing the creation of complex functionalities such as:

1. **Chains**: Sequences of operations or workflows.
2. **Agents**: Intelligent components that perform specific tasks.
3. **Memory**: Capabilities to handle state or previous interactions.
4. **Retrievers**: Components for fetching information from various sources.

In essence, LangChain facilitates the construction of applications that leverage AI tools to execute diverse and dynamic tasks.


### Semantic chunker With Langchain

In [34]:
from langchain_ollama import OllamaEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import TextLoader

In [37]:
## Load the documents
loader = TextLoader("02. langchain_intro.txt")
documents = loader.load()

## Initialize embedding model
embeddings = OllamaEmbeddings(
    model = "qwen3-embedding:8b"
)

## Create the semantic chunker
chunker = SemanticChunker(embeddings, breakpoint_threshold_type="percentile", breakpoint_threshold_amount=8.0)

## Split the documents
chunks = chunker.split_documents(documents)

print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:\n{chunk.page_content}\n")

Number of chunks: 4
Chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

Chunk 2:
You can create chains, agents, memory, and retrievers.

Chunk 3:
The Eiffel Tower is located in Paris.

Chunk 4:
France is a popular tourist destination.

